<a href="https://colab.research.google.com/github/bradfa/colab-notebooks/blob/main/qwen3.5-35b-a3b_kv_cache_kld_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KV-Cache Quantization KLD Benchmark

Measures the impact of different KV-cache quantization approaches on
**Qwen3.5-35B-A3B** using Kullback-Leibler Divergence (KLD),
perplexity ratio, and same-top-p metrics via llama.cpp's `llama-perplexity`.

## Design
- **Model**: Qwen3.5-35B-A3B — mixture-of-experts transformer, 35B total / ~3B active
  parameters per forward pass, Apache 2.0
- **Weights**: GGUFs from `unsloth/Qwen3.5-35B-A3B-GGUF` (BF16, Q8_0, Q4_K_M) and
  `byteshape/Qwen3.5-35B-A3B-GGUF` (IQ4_XS-4.12bpw); BF16 is a 2-shard split (~69.4 GB)
- **Corpus**: Wikitext-2 test set (64 chunks)
- **Baseline**: BF16 weights + BF16 KV cache → logit dump file (~33 GB)
- **Sweep**: 4 weight quants × 4 KV cache types (bf16, f16, q8_0, q4_0)
- **Binary**: llama.cpp built from source with CUDA support
- **Runtime**: NVIDIA RTX Pro 6000 96 GB (Colab G4)

## Runtime estimate
| GPU | Chunks | Approx total time |
|-----|--------|-------------------|
| RTX Pro 6000 96 GB | 64 | ~60-90 min |

## Note on TurboQuant
TQ3/TQ4 (Zandieh et al., ICLR 2026) are not yet merged into mainline
llama.cpp and therefore unavailable in prebuilt binaries. Noted as future work.

## 0 — Configuration
Edit these if you want to customise the run.

In [ ]:
# ---- user-editable configuration ----

# Weight quantization GGUFs to sweep (label, hf_repo_id, filename).
# For split GGUFs, filename is a list of shard paths (relative to WORK_DIR).
# llama.cpp receives the first shard path and auto-loads the rest.
WEIGHT_CONFIGS = [
    ("BF16",   "unsloth/Qwen3.5-35B-A3B-GGUF", [
                   "BF16/Qwen3.5-35B-A3B-BF16-00001-of-00002.gguf",
                   "BF16/Qwen3.5-35B-A3B-BF16-00002-of-00002.gguf",
               ]),
    ("Q8_0",   "unsloth/Qwen3.5-35B-A3B-GGUF",  "Qwen3.5-35B-A3B-Q8_0.gguf"),
    ("Q4_K_M", "unsloth/Qwen3.5-35B-A3B-GGUF",  "Qwen3.5-35B-A3B-Q4_K_M.gguf"),
    ("IQ4_XS", "byteshape/Qwen3.5-35B-A3B-GGUF", "Qwen3.5-35B-A3B-IQ4_XS-4.12bpw.gguf"),
]

# KV cache types to sweep  (each entry is a (key_type, value_type) tuple)
# Note: q5_1, q5_0, q4_1, iq4_nl fall back to CPU and are excluded.
# Asymmetric mixed K/V types (e.g. q8_0/q4_0) also fall back to CPU.
KV_CONFIGS = [
    ("bf16", "bf16"),
    ("f16",  "f16"),
    ("q8_0", "q8_0"),
    ("q4_0", "q4_0"),
]

# baseline KV cache type (used for the logit dump) — always BF16 weights + BF16 KV
BASELINE_K = "bf16"
BASELINE_V = "bf16"

# number of chunks (fixed for RTX Pro 6000 96 GB)
N_CHUNKS = 64

# context size per chunk (llama.cpp default is 512; match for comparability)
CTX_SIZE = 512

# number of GPU layers to offload (-1 = all)
N_GPU_LAYERS = 99

# paths
WORK_DIR      = "/content/kv_bench"
LOGITS_PATH   = f"{WORK_DIR}/baseline_logits.kld"
WIKITEXT_PATH = f"{WORK_DIR}/wikitext-2-raw/wiki.test.raw"
LLAMA_CPP_DIR = f"{WORK_DIR}/llama.cpp"
BIN_DIR       = f"{WORK_DIR}/bin"
RESULTS_CSV   = f"{WORK_DIR}/kld_results.csv"


def gguf_path_for(filename):
    """Return the model path to pass to llama.cpp (first shard for split GGUFs)."""
    first = filename[0] if isinstance(filename, list) else filename
    return f"{WORK_DIR}/{first}"


BASELINE_GGUF = gguf_path_for(WEIGHT_CONFIGS[0][2])

## 1 — GPU Sanity Check

In [ ]:
import subprocess, os, re, shutil

os.makedirs(WORK_DIR, exist_ok=True)

try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total",
         "--format=csv,noheader,nounits"],
        text=True
    ).strip()
    GPU_NAME, gpu_mem_str = out.split(",")
    GPU_NAME = GPU_NAME.strip()
    GPU_MEM_MB = int(gpu_mem_str.strip())
except Exception as e:
    raise RuntimeError(
        f"No NVIDIA GPU detected: {e}. Enable GPU in Runtime > Change runtime type."
    )

driver = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
    text=True
).strip()

print(f"GPU    : {GPU_NAME}")
print(f"VRAM   : {GPU_MEM_MB} MB")
print(f"Driver : {driver}")
print(f"Chunks : {N_CHUNKS}")

## 2 — Build llama.cpp from Source

Clones the llama.cpp repository and builds `llama-perplexity` with CUDA support.
This takes approximately 5-10 minutes on the RTX Pro 6000.

In [ ]:
import glob

os.makedirs(BIN_DIR, exist_ok=True)

PERPLEXITY_BIN = f"{BIN_DIR}/llama-perplexity"


def build_from_source():
    """Clone llama.cpp and build llama-perplexity with CUDA support."""
    print("Building llama.cpp from source...")
    if not os.path.isdir(LLAMA_CPP_DIR):
        subprocess.run(
            ["git", "clone", "--depth=1",
             "https://github.com/ggml-org/llama.cpp", LLAMA_CPP_DIR],
            check=True
        )
    build_dir = f"{LLAMA_CPP_DIR}/build"
    subprocess.run(
        ["cmake", LLAMA_CPP_DIR, "-B", build_dir,
         "-DGGML_CUDA=ON", "-DBUILD_SHARED_LIBS=OFF",
         "-DCMAKE_BUILD_TYPE=Release"],
        check=True
    )
    n_jobs = os.cpu_count() or 4
    subprocess.run(
        ["cmake", "--build", build_dir, "--config", "Release",
         "-j", str(n_jobs), "--target", "llama-perplexity"],
        check=True
    )
    src = f"{build_dir}/bin/llama-perplexity"
    if os.path.isfile(src):
        shutil.copy2(src, PERPLEXITY_BIN)
    for so in glob.glob(f"{build_dir}/bin/*.so") + glob.glob(f"{build_dir}/src/*.so"):
        shutil.copy2(so, BIN_DIR)
    print("  Source build complete.")


if not os.path.isfile(PERPLEXITY_BIN):
    build_from_source()

result = subprocess.run([PERPLEXITY_BIN, "--help"], capture_output=True, text=True)
if result.returncode not in (0, 1):
    raise RuntimeError(f"llama-perplexity not working: {result.stderr[:500]}")
print(f"\nllama-perplexity ready at {PERPLEXITY_BIN}")

## 3 — Download Model GGUFs

Downloads all weight-quantization GGUFs:
- **BF16** (2-shard split, ~69.4 GB total) from `unsloth/Qwen3.5-35B-A3B-GGUF`
- **Q8_0** (~36.9 GB) from `unsloth/Qwen3.5-35B-A3B-GGUF`
- **Q4_K_M** (~22 GB) from `unsloth/Qwen3.5-35B-A3B-GGUF`
- **IQ4_XS-4.12bpw** (~17.9 GB) from `byteshape/Qwen3.5-35B-A3B-GGUF`

The BF16 first shard also serves as the baseline for the logit dump in section 5.

In [ ]:
subprocess.run(
    ["pip", "install", "-q", "huggingface_hub[hf_transfer]"],
    check=True
)

from huggingface_hub import hf_hub_download
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

os.makedirs(WORK_DIR, exist_ok=True)

for label, repo_id, filename in WEIGHT_CONFIGS:
    filenames = filename if isinstance(filename, list) else [filename]
    for fn in filenames:
        dest = f"{WORK_DIR}/{fn}"
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        if os.path.isfile(dest):
            print(f"{label} ({fn}): already present ({os.path.getsize(dest)/1e9:.2f} GB)")
        else:
            print(f"Downloading {label} ({fn}) from {repo_id}...")
            downloaded = hf_hub_download(
                repo_id=repo_id,
                filename=fn,
                local_dir=WORK_DIR,
            )
            if os.path.abspath(downloaded) != os.path.abspath(dest):
                os.rename(downloaded, dest)
            print(f"  Saved: {os.path.getsize(dest)/1e9:.2f} GB")

## 4 — Download Wikitext-2 Test Set

In [ ]:
if not os.path.isfile(WIKITEXT_PATH):
    print("Downloading Wikitext-2 test set from HuggingFace datasets...")
    subprocess.run(
        ["pip", "install", "-q", "datasets"],
        check=True
    )
    from datasets import load_dataset

    os.makedirs(os.path.dirname(WIKITEXT_PATH), exist_ok=True)

    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    with open(WIKITEXT_PATH, "w", encoding="utf-8") as f:
        for row in ds:
            f.write(row["text"])

    wt_size = os.path.getsize(WIKITEXT_PATH)
    print(f"Wikitext-2 test set: {WIKITEXT_PATH}")
    print(f"Size: {wt_size / 1e6:.1f} MB")
else:
    print(f"Wikitext-2 already present: {WIKITEXT_PATH}")

## 5 — Baseline Logit Dump

Run perplexity with BF16 weights and BF16 KV cache, saving the full logit
distributions to a binary file. This is the reference for all KLD comparisons.

With Qwen3.5-35B-A3B's 248,320-token vocabulary and 64 chunks of 512 context,
the logit file is approximately 33 GB. Each subsequent KLD run reads this file
but does **not** produce additional large files.

In [ ]:
import time

def run_perplexity(cache_type_k, cache_type_v, kld_mode="dump", label="", gguf_path=None):
    """Run llama-perplexity and return (stdout, stderr, elapsed_seconds).

    kld_mode:
      "dump"    -> baseline run, writes logits to LOGITS_PATH
      "compare" -> KLD run, reads LOGITS_PATH and computes divergence
      "none"    -> plain perplexity, no KLD
    """
    model = gguf_path if gguf_path is not None else BASELINE_GGUF
    cmd = [
        PERPLEXITY_BIN,
        "-m", model,
        "-f", WIKITEXT_PATH,
        "-ngl", str(N_GPU_LAYERS),
        "--chunks", str(N_CHUNKS),
        "-c", str(CTX_SIZE),
        "--flash-attn", "on",
        "-ctk", cache_type_k,
        "-ctv", cache_type_v,
    ]
    if kld_mode == "dump":
        cmd += ["--kl-divergence-base", LOGITS_PATH]
    elif kld_mode == "compare":
        cmd += ["--kl-divergence-base", LOGITS_PATH, "--kl-divergence"]

    tag = label or f"ctk={cache_type_k} ctv={cache_type_v}"
    print(f"\n{'='*60}")
    print(f"Running: {tag}  (mode={kld_mode})")
    print(f"{'='*60}")

    env = os.environ.copy()
    env["LD_LIBRARY_PATH"] = BIN_DIR + ":" + env.get("LD_LIBRARY_PATH", "")

    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, env=env)
    elapsed = time.time() - t0

    stdout = proc.stdout.decode("utf-8", errors="replace")
    stderr = proc.stderr.decode("utf-8", errors="replace")

    if proc.returncode != 0:
        print(f"WARNING: non-zero return code {proc.returncode}")
        print(f"stderr (last 1000 chars):\n{stderr[-1000:]}")

    lines = stdout.strip().split("\n")
    for line in lines[-30:]:
        print(line)

    print(f"\nElapsed: {elapsed:.1f}s")
    return stdout, stderr, elapsed


# --- run baseline ---
est_gb = N_CHUNKS * CTX_SIZE * 248320 * 4 / 1e9
print(f"Estimated logit file size: {est_gb:.1f} GB")

if os.path.isfile(LOGITS_PATH):
    print(f"Baseline logits already exist at {LOGITS_PATH}")
    print(f"Size: {os.path.getsize(LOGITS_PATH) / 1e9:.2f} GB")
    print("Delete this file to regenerate.")
else:
    stdout, stderr, elapsed = run_perplexity(
        BASELINE_K, BASELINE_V,
        kld_mode="dump",
        label=f"BASELINE ctk={BASELINE_K} ctv={BASELINE_V}",
        gguf_path=BASELINE_GGUF,
    )
    if os.path.isfile(LOGITS_PATH):
        print(f"\nBaseline logits saved: {os.path.getsize(LOGITS_PATH) / 1e9:.2f} GB")
    else:
        print("\nERROR: logit file was not created. Check stderr above.")

## 6 — KLD Sweep

For each weight quantization × KV cache configuration, run
`llama-perplexity --kl-divergence` against the baseline logits and parse the output.

In [ ]:
import pandas as pd

def parse_kld_output(stdout):
    """Extract KLD metrics from llama-perplexity stdout.

    Handles the output format:
      Mean    KLD:   0.000312 ±   0.000004
      Same top p: 99.136 ± 0.072 %
    """
    metrics = {}
    for line in stdout.split("\n"):
        line = line.strip()
        # Mean KLD line: "Mean    KLD:   0.000312 ±   0.000004"
        if re.match(r"Mean\s+KLD:", line):
            m = re.search(r"([\d.]+(?:e[+-]?\d+)?)\s*±\s*([\d.]+(?:e[+-]?\d+)?)", line)
            if m:
                metrics["kld_mean"] = float(m.group(1))
                metrics["kld_stderr"] = float(m.group(2))
        # Same top p: "Same top p: 99.136 ± 0.072 %"
        elif re.match(r"Same top p:", line):
            m = re.search(r"([\d.]+)\s*±", line)
            if m:
                metrics["same_top_p_pct"] = float(m.group(1))
        # PPL ratio if present
        elif "ppl ratio" in line.lower() or "perplexity ratio" in line.lower():
            m = re.search(r"([\d.]+(?:e[+-]?\d+)?)", line)
            if m:
                metrics["ppl_ratio"] = float(m.group(1))
        # Final perplexity estimate
        elif line.startswith("Final estimate:") or ("perplexity" in line.lower() and "final" in line.lower()):
            m = re.search(r"([\d.]+)\s*\+/-", line)
            if m:
                metrics["ppl"] = float(m.group(1))
    return metrics


results = []

for (wlabel, wrepo, wfile) in WEIGHT_CONFIGS:
    wgguf = gguf_path_for(wfile)
    for (ctk, ctv) in KV_CONFIGS:
        label = f"{wlabel} K:{ctk} V:{ctv}"
        stdout, stderr, elapsed = run_perplexity(
            ctk, ctv, kld_mode="compare", label=label, gguf_path=wgguf
        )
        metrics = parse_kld_output(stdout)
        metrics["weight_quant"] = wlabel
        metrics["cache_type_k"] = ctk
        metrics["cache_type_v"] = ctv
        metrics["kv_label"] = f"K:{ctk} V:{ctv}"
        metrics["elapsed_s"] = round(elapsed, 1)
        results.append(metrics)

        if "kld_mean" in metrics:
            print(f"  -> KLD mean: {metrics['kld_mean']:.6f}  same_top_p: {metrics.get('same_top_p_pct', 'n/a')}%")
        else:
            print("  -> WARNING: could not parse KLD from output")

df = pd.DataFrame(results)
df.to_csv(RESULTS_CSV, index=False)
print(f"\nResults saved to {RESULTS_CSV}")
print(df.to_string(index=False))

## 7 — Results Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
matplotlib.rcParams.update({"font.size": 11})

df = pd.read_csv(RESULTS_CSV)

if "kld_mean" not in df.columns or df["kld_mean"].isna().all():
    print("No KLD data to plot. Check the raw output above for errors.")
else:
    weight_order = [w for w, _, _ in WEIGHT_CONFIGS]
    kv_order = [f"K:{k} V:{v}" for k, v in KV_CONFIGS]

    kld_pivot = df.pivot(index="kv_label", columns="weight_quant", values="kld_mean")
    kld_pivot = kld_pivot.reindex(index=kv_order, columns=weight_order)

    same_pivot = df.pivot(index="kv_label", columns="weight_quant", values="same_top_p_pct")
    same_pivot = same_pivot.reindex(index=kv_order, columns=weight_order)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle(
        f"KV-Cache x Weight Quantization Impact -- Qwen3.5-35B-A3B\n"
        f"Baseline: BF16 weights, K={BASELINE_K} V={BASELINE_V} | {N_CHUNKS} chunks | {GPU_NAME}",
        fontsize=13, fontweight="bold"
    )

    # KLD heatmap
    ax = axes[0]
    im = ax.imshow(kld_pivot.values.astype(float), aspect="auto", cmap="YlOrRd")
    ax.set_xticks(range(len(weight_order)))
    ax.set_xticklabels(weight_order, rotation=45, ha="right")
    ax.set_yticks(range(len(kv_order)))
    ax.set_yticklabels(kv_order)
    ax.set_title("Mean KLD (lower = better)")
    plt.colorbar(im, ax=ax)
    for i in range(len(kv_order)):
        for j in range(len(weight_order)):
            val = kld_pivot.values[i, j]
            if not np.isnan(float(val)):
                ax.text(j, i, f"{float(val):.4f}", ha="center", va="center", fontsize=9)

    # Same top p heatmap
    ax = axes[1]
    im2 = ax.imshow(same_pivot.values.astype(float), aspect="auto", cmap="YlGn")
    ax.set_xticks(range(len(weight_order)))
    ax.set_xticklabels(weight_order, rotation=45, ha="right")
    ax.set_yticks(range(len(kv_order)))
    ax.set_yticklabels(kv_order)
    ax.set_title("Same Top Token % (higher = better)")
    plt.colorbar(im2, ax=ax)
    for i in range(len(kv_order)):
        for j in range(len(weight_order)):
            val = same_pivot.values[i, j]
            if not np.isnan(float(val)):
                ax.text(j, i, f"{float(val):.1f}%", ha="center", va="center", fontsize=9)

    plt.tight_layout()
    plt.savefig(f"{WORK_DIR}/kld_results.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Plot saved to {WORK_DIR}/kld_results.png")

## 8 — Summary Table

In [ ]:
if "kld_mean" in df.columns:
    display_cols = [c for c in ["weight_quant", "kv_label", "kld_mean", "kld_stderr",
                                 "same_top_p_pct", "elapsed_s"] if c in df.columns]
    summary = df[display_cols].copy()
    col_names = {
        "weight_quant": "Weights", "kv_label": "KV Cache",
        "kld_mean": "KLD Mean", "kld_stderr": "KLD StdErr",
        "same_top_p_pct": "Same Top %", "elapsed_s": "Time (s)"
    }
    summary.rename(columns=col_names, inplace=True)
    print(summary.to_markdown(index=False, floatfmt=".6f"))
else:
    print("No results to summarize.")

## 9 — Cleanup (optional)

The baseline logit file is approximately 33 GB. Uncomment the cell below to delete it.

In [ ]:
# if os.path.isfile(LOGITS_PATH):
#     os.remove(LOGITS_PATH)
#     print(f"Deleted {LOGITS_PATH}")

## Notes

### Interpreting the results
- **KL Divergence (KLD)**: measures how much the quantized KV-cache (or quantized weights)
  shifts the output probability distribution vs the BF16 baseline. Lower is better.
  A value of 0 means identical distributions.
- **PPL Ratio**: ratio of quantized perplexity to baseline perplexity.
  Values close to 1.0 mean negligible quality loss.
- **Same Top %**: how often the most-probable token is the same between
  baseline and quantized. Higher is better.

### Model characteristics
Qwen3.5-35B-A3B is a mixture-of-experts model. Only approximately 3B parameters are
active per forward pass, but all 35B must be loaded into VRAM. The large vocabulary
(248,320 tokens) and GQA attention mean both weight quantization and KV cache
quantization may show different sensitivity profiles than dense models of similar
active-parameter count.

### Key findings from prior work
- q8_0 KV cache typically shows negligible quality loss vs FP16/BF16.
- Models with aggressive GQA (fewer KV heads) may be more sensitive to KV quantization.

### Planned extensions
- **TurboQuant (TQ3/TQ4)**: once merged into mainline llama.cpp, add
  ("tq3", "tq3") and ("tq4", "tq4") to KV_CONFIGS.
- **Code-focused corpus**: test on C or Go source to see if KV cache quantization
  affects code generation differently than prose.